# Chapter 3 (leptonic) — Notebook 3: Building m(ℓνb)

**Goals**

- Combine the lepton, reconstructed neutrino and leptonic b-jet.
- Plot the reconstructed top-mass distribution and compare to the generator value (≈172.5 GeV).

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

In [ ]:
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)
events = events[selection.semilep_preselection(events, cuts)]

lep = kinematics.leading_lepton(events)
met = kinematics.met_vector(events)
jets = kinematics.jet_vectors(events)
is_b = events.jet_btag_quantile >= cuts.btag_quantile_min
b_jets = jets[is_b]
keep = ak.num(b_jets) >= 2
events, lep, met, b_jets = events[keep], lep[keep], met[keep], b_jets[keep][:, :2]

nu = neutrino.build_neutrino(lep, met)
b_lep, _ = pairing.assign_bjets(b_jets, lep)
m_top_lep = (lep + nu + b_lep).mass

plt.hist(ak.to_numpy(m_top_lep), bins=60, range=(80, 350))
plt.axvline(172.5, color='red', label='generator $m_t$ = 172.5 GeV')
plt.xlabel(r'$m(\ell, \nu_{\rm reco}, b_{\rm lep})$ [GeV]'); plt.legend()

## ✏️ Your turn 3.1

▶️ Change the neutrino-root selector and re-run.

The W-mass quadratic gives two $p_z(\nu)$ roots. `SELECTOR` chooses which one to use:
`'smallest_abs'` (default), `'plus'`, or `'minus'`. Re-run for each and read off the peak width σ
printed below — which choice gives the **narrowest** top-mass peak?

> **Stretch (optional):** compare all three selectors and decide which the analysis should use.

In [ ]:
SELECTOR = 'smallest_abs'    # ✏️ try 'smallest_abs', 'plus', 'minus'

nu = neutrino.build_neutrino(lep, met, selector=SELECTOR)
m_top = ak.to_numpy((lep + nu + b_lep).mass)

counts, edges = np.histogram(m_top, bins=50, range=(80, 350))
result = fitting.fit_gaussian(counts.astype(float), edges,
                              p0={'n': counts.sum(), 'mu': 172.5, 'sigma': 30.0})
print(f"selector '{SELECTOR}':  peak = {result.params['mu']:.1f} GeV,  width σ = {result.params['sigma']:.1f} GeV")

plt.hist(m_top, bins=50, range=(80, 350), histtype='step', label=SELECTOR)
plt.axvline(172.5, color='red', label='generator $m_t$ = 172.5 GeV')
plt.xlabel(r'$m(\ell\nu b)$ [GeV]'); plt.ylabel('Events'); plt.legend()